# Scaled Final Validation: Bigger N, Denser Layer Sweep, Permutation Control

The deep-layer raw-signal finding has now replicated once (two independent runs, both
showing layer 24 clearly beating the entropy baseline). This notebook is the final
validation pass before writing begins:

1. **N=400 instead of N=200** -- more statistical power behind the headline claim, a third
   independent batch (fresh seeds throughout).
2. **Five layers instead of three** (6, 12, 18, 24, 30) -- reading extra layers from the
   same forward pass is nearly free, so this properly maps out *where* the signal peaks
   across depth instead of relying on three arbitrarily-chosen points.
3. **A permutation control** -- for the best-performing feature set, refit the exact same
   classifier on the exact same features but with the pass/fail labels randomly shuffled.
   If the real result is genuine pattern-matching (not an artifact of the classifier or the
   PCA step), the shuffled version should collapse to chance (~0.50 ROC-AUC). This is a
   standard, cheap sanity check reviewers expect for exactly this kind of claim.

All generation is natural/unsteered -- there is no steering vector anywhere in this notebook.
This is intended as the last big run before the paper write-up starts.

Kaggle setup: Accelerator = **GPU T4 x1 or x2**, Internet = **ON**. Expect ~50-70 minutes.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import gc
import math
import collections
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, EsmForProteinFolding

torch.manual_seed(7213)
np.random.seed(7213)

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def calculate_entropy(seq_str):
    if not seq_str:
        return 0.0
    counts = collections.Counter(seq_str)
    total = len(seq_str)
    return -sum((c / total) * math.log2(c / total) for c in counts.values())

# Simple, established composition-based intrinsic-disorder proxy (Dunker/Uversky-style
# disorder-promoting vs. order-promoting residue sets). This is a coarse heuristic, not a
# validated disorder predictor -- treat it as a sanity check, not ground truth.
DISORDER_PROMOTING = set("ARGQSPEK")
ORDER_PROMOTING = set("WCFIYVLN")

def disorder_score(seq):
    if not seq:
        return 0.0
    n = len(seq)
    d = sum(1 for a in seq if a in DISORDER_PROMOTING) / n
    o = sum(1 for a in seq if a in ORDER_PROMOTING) / n
    return d - o

print("Setup complete. CUDA available:", torch.cuda.is_available())

Setup complete. CUDA available: True


In [2]:
class OrderedSAE(nn.Module):
    def __init__(self, d_model=1280, expansion_factor=4, k=16, nesting_list=[64, 256, 1024, 5120]):
        super().__init__()
        self.d_latent = d_model * expansion_factor
        self.k = k
        self.nesting_list = nesting_list
        self.encoder = nn.Linear(d_model, self.d_latent)
        self.decoder = nn.Linear(self.d_latent, d_model)

    def forward(self, x):
        acts = F.relu(self.encoder(x))
        topk_vals, topk_idx = torch.topk(acts, self.k, dim=-1)
        sparse_acts = torch.zeros_like(acts).scatter_(-1, topk_idx, topk_vals)
        reconstructions = []
        for nest_size in self.nesting_list:
            mask = torch.zeros_like(sparse_acts)
            mask[:, :nest_size] = 1.0
            reconstructions.append(self.decoder(sparse_acts * mask))
        return reconstructions, sparse_acts


class PLMExtractor:
    def __init__(self, target_layer=12):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Loading ProtGPT2 on {self.device}...")
        self.tokenizer = AutoTokenizer.from_pretrained("nferruz/ProtGPT2")
        self.model = AutoModelForCausalLM.from_pretrained("nferruz/ProtGPT2").to(self.device)
        self.target_layer = target_layer

In [3]:
# --- Train the OSAE once more, kept only as the "layer-12, sequence-level latents" arm
#     for continuity with earlier runs ---

positive_seqs = [
    "NLYIQWLKDGGPSSGRPPPS", "LSDEDFKAVFGMTRSAFANLPLWKQQHLKKEKGLF", "GSQIGAKNTGQVQLNLLAL",
    "MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE",
    "MKTIIALSYIFCLVFADYKDDDDKLEHTHHHEASGGNLQVQLQESGGGLVQAGGSLRLSCAASGRTFSNYAMGWFRQAPGKEREFVAAISWSGGSTYYTDSVKGRFTISRDNAKNTVYLQMNSLKPEDTAVYYCAASRFRYWGQGTQVTVSS",
    "DEPPQSPWDRVKDFATVYVDAVKPTGKGKV",
]
degenerate_seqs = [
    "AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA", "LGLGLGLGLGLGLGLGLGLGLGLGLGLGLGLGLGL",
    "GGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGG", "SSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSS",
    "PGPGPGPGPGPGPGPGPGPGPGPGPGPGPGPGPGP", "QWQWQWQWQWQWQWQWQWQWQWQWQWQWQWQWQWQ",
]

def get_mean_activation(extractor, seq_list, layer):
    acts = []
    for seq in seq_list:
        inputs = extractor.tokenizer(seq, return_tensors="pt").to(extractor.device)
        with torch.no_grad():
            out = extractor.model(**inputs, output_hidden_states=True)
            acts.append(out.hidden_states[layer].mean(dim=1).squeeze(0).cpu())
    return torch.stack(acts)

def train_ordered_sae(X_train, device, epochs=1000, lr=1e-3):
    osae = OrderedSAE(d_model=X_train.shape[1], expansion_factor=4, k=16).to(device)
    optimizer = torch.optim.Adam(osae.parameters(), lr=lr)
    X_train = X_train.to(device)
    osae.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        reconstructions, _ = osae(X_train)
        loss = sum(F.mse_loss(recon, X_train) for recon in reconstructions)
        loss.backward()
        optimizer.step()
    return osae

print("=== Training OSAE (layer-12 reference arm only, no steering) ===")
extractor = PLMExtractor(target_layer=12)

pos_acts = get_mean_activation(extractor, positive_seqs, extractor.target_layer)
neg_acts = get_mean_activation(extractor, degenerate_seqs, extractor.target_layer)
X_train_mixed = torch.cat([pos_acts, neg_acts], dim=0).to(extractor.device)
osae_model = train_ordered_sae(X_train_mixed, device=extractor.device)

v_L = (pos_acts.mean(dim=0) - neg_acts.mean(dim=0)).to(extractor.device)
osae_model.eval()
with torch.no_grad():
    acts = F.relu(osae_model.encoder(v_L.unsqueeze(0)))
    topk_vals, topk_idx = torch.topk(acts, osae_model.k, dim=-1)
    sparse_v_L = torch.zeros_like(acts).scatter_(-1, topk_idx, topk_vals).squeeze(0)

active_indices = torch.nonzero(sparse_v_L).squeeze(-1).cpu().numpy()
active_weights = sparse_v_L[active_indices].detach().cpu().numpy()
order = np.argsort(-np.abs(active_weights))
active_indices = active_indices[order]

TOP_K_CAUSAL = 3
causal_indices = active_indices[:TOP_K_CAUSAL].tolist()
print(f"Layer-12 OSAE top-{TOP_K_CAUSAL} latents (reference arm): {causal_indices}")

=== Training OSAE (layer-12 reference arm only, no steering) ===
Loading ProtGPT2 on cuda...


config.json:   0%|          | 0.00/850 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: nferruz/ProtGPT2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Layer-12 OSAE top-3 latents (reference arm): [4380, 1603, 1677]


In [4]:
# --- Same real, UniProt-derived calibration prefixes ---
import urllib.request

UNIPROT_ACCESSIONS = [
    "P0CG48", "P00720", "P02144", "P42212", "P01308", "P61823",
    "P00648", "P99999", "P69905", "P68871", "P00698", "P00441",
]

def fetch_uniprot_sequence(accession, timeout=10):
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.fasta"
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:
            text = resp.read().decode("utf-8")
        lines = [l for l in text.strip().split("\n") if l]
        seq = "".join(lines[1:])
        return seq if len(seq) >= 20 else None
    except Exception as e:
        print(f"  skip {accession}: {e}")
        return None

print("Fetching real reference protein sequences from UniProt...")
reference_seqs = []
for acc in UNIPROT_ACCESSIONS:
    seq = fetch_uniprot_sequence(acc)
    if seq:
        reference_seqs.append((acc, seq))
        print(f"  fetched {acc}: {len(seq)} residues")

if len(reference_seqs) < 5:
    raise RuntimeError("Fewer than 5 reference sequences fetched -- check Kaggle internet access is ON.")

def build_prefix_pool(reference_seqs, n_prefixes, min_len=10, max_len=15, seed=305):
    rng = np.random.RandomState(seed)
    prefixes = []
    for i in range(n_prefixes):
        acc, seq = reference_seqs[i % len(reference_seqs)]
        plen = rng.randint(min_len, max_len + 1)
        start = rng.randint(0, max(1, len(seq) - plen))
        prefixes.append(seq[start:start + plen])
    return prefixes

calibration_prefixes = build_prefix_pool(reference_seqs, n_prefixes=400)
print(f"Built {len(calibration_prefixes)} calibration prefixes from {len(reference_seqs)} reference proteins.")

Fetching real reference protein sequences from UniProt...
  fetched P0CG48: 685 residues
  fetched P00720: 164 residues
  fetched P02144: 154 residues
  fetched P42212: 238 residues
  fetched P01308: 110 residues
  fetched P61823: 150 residues
  fetched P00648: 157 residues
  fetched P99999: 105 residues
  fetched P69905: 142 residues
  fetched P68871: 147 residues
  fetched P00698: 147 residues
  fetched P00441: 154 residues
Built 400 calibration prefixes from 12 reference proteins.


In [5]:
# --- Simple natural generation (no hooks needed -- we'll re-encode afterward for features) ---

def generate_natural(extractor, prompts, max_len=50, seed=0):
    torch.manual_seed(seed)
    extractor.model.eval()
    device = next(extractor.model.parameters()).device
    records = []
    for prompt in prompts:
        inputs = extractor.tokenizer(prompt, return_tensors="pt").to(device)
        prompt_len = inputs["input_ids"].shape[1]
        with torch.no_grad():
            output_ids = extractor.model.generate(
                **inputs, max_length=max_len, do_sample=True,
                temperature=1.2, pad_token_id=extractor.tokenizer.eos_token_id
            )
        full_seq = extractor.tokenizer.decode(output_ids[0], skip_special_tokens=True).replace(" ", "")
        records.append({
            "prompt": prompt,
            "prompt_len": prompt_len,
            "output_ids": output_ids[0].cpu().tolist(),
            "sequence": full_seq,
        })
    clear_gpu()
    return records

print("=== Generating N=400 fully NATURAL sequences ===")
natural_records = generate_natural(extractor, calibration_prefixes, max_len=50, seed=1717)
print(f"Generated {len(natural_records)} natural sequences.")

=== Generating N=400 fully NATURAL sequences ===
Generated 400 natural sequences.


In [6]:
# --- One teacher-forced forward pass per sequence -> hidden states at ALL layers at once.
#     This is mathematically identical to what the model saw during generation (the tokens
#     are already fixed/sampled), just computed in a single non-autoregressive pass. ---

RAW_LAYERS = [6, 12, 18, 24, 30]
WINDOW = 15

def extract_features(extractor, osae, records, raw_layers, window=WINDOW):
    device = next(extractor.model.parameters()).device
    for r in records:
        ids_tensor = torch.tensor([r["output_ids"]], device=device)
        with torch.no_grad():
            out = extractor.model(ids_tensor, output_hidden_states=True)
        prompt_len = r["prompt_len"]
        seq_len = ids_tensor.shape[1]
        T = min(window, max(0, seq_len - prompt_len))

        # raw hidden states at each candidate layer
        for L in raw_layers:
            hs = out.hidden_states[L][0]  # [seq_len, d_model]
            feat = hs[prompt_len:prompt_len + T].sum(dim=0).cpu().numpy() if T > 0 else np.zeros(hs.shape[-1])
            r[f"raw_layer_{L}"] = feat

        # layer-12 hidden states projected through the OSAE encoder (reference arm)
        hs12 = out.hidden_states[12][0]
        with torch.no_grad():
            osae_acts = F.relu(osae.encoder(hs12[prompt_len:prompt_len + T])) if T > 0 else torch.zeros(0, osae.d_latent, device=device)
        r["osae_layer12_full"] = osae_acts.sum(dim=0).cpu().numpy() if T > 0 else np.zeros(osae.d_latent)

        # token-level entropy trajectory (generated part only), cumulative
        ent_sum = 0.0
        ent_vals = []
        for t in range(1, T + 1):
            partial_ids = r["output_ids"][: prompt_len + t]
            partial_str = extractor.tokenizer.decode(partial_ids, skip_special_tokens=True).replace(" ", "")
            gen_only = partial_str[len(r["prompt"]):] if partial_str.startswith(r["prompt"]) else partial_str
            ent_vals.append(calculate_entropy(gen_only))
        r["entropy_cumsum"] = float(np.sum(ent_vals)) if ent_vals else 0.0
    return records

print(f"Extracting features at layers {RAW_LAYERS} (+ layer-12 OSAE projection) via teacher-forced re-encoding...")
natural_records = extract_features(extractor, osae_model, natural_records, RAW_LAYERS, WINDOW)
print("Done.")

Extracting features at layers [6, 12, 18, 24, 30] (+ layer-12 OSAE projection) via teacher-forced re-encoding...
Done.


In [7]:
print("=== Freeing ProtGPT2 + OSAE from GPU ===")
del extractor
del osae_model
clear_gpu()
print(f"GPU memory allocated after purge: {torch.cuda.memory_allocated()/1e9:.3f} GB")

=== Freeing ProtGPT2 + OSAE from GPU ===
GPU memory allocated after purge: 0.018 GB


In [8]:
VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

class StructuralEvaluator:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print("Loading ESMFold...")
        self.tokenizer = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
        self.model = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1", low_cpu_mem_usage=True)
        self.model = self.model.to(self.device).eval()

    def fold_one(self, seq):
        cleaned = "".join(a for a in seq if a in VALID_AA)
        if len(cleaned) < 10:
            return 0.0, None, 0.0
        inputs = self.tokenizer([cleaned], return_tensors="pt", add_special_tokens=False).to(self.device)
        t0 = time.time()
        plddt, ptm = 0.0, None
        try:
            with torch.no_grad():
                out = self.model(**inputs)
            raw = float(np.mean(out.plddt.cpu().numpy()))
            plddt = raw * 100.0 if raw <= 1.5 else raw
            if hasattr(out, "ptm") and out.ptm is not None:
                ptm = float(out.ptm.flatten()[0].item()) if torch.is_tensor(out.ptm) else float(out.ptm)
        except RuntimeError:
            clear_gpu()
        except Exception as e:
            print(f"  (ptm read failed, continuing without it: {e})")
        dt = time.time() - t0
        return plddt, ptm, dt

evaluator = StructuralEvaluator()

def fold_records(records, evaluator):
    for r in records:
        plddt, ptm, dt = evaluator.fold_one(r["sequence"])
        r["plddt"] = plddt
        r["ptm"] = ptm
        r["fold_time_s"] = dt
        r["collapse_60"] = int(0.0 < plddt < 60.0)
        r["collapse_70"] = int(0.0 < plddt < 70.0)
    return records

print("Folding natural sequences...")
natural_records = fold_records(natural_records, evaluator)

del evaluator
clear_gpu()

plddts = [r["plddt"] for r in natural_records if r["plddt"] > 0.0]
ptms = [r["ptm"] for r in natural_records if r["ptm"] is not None]
print(f"\nMean pLDDT: {np.mean(plddts):.2f}")
if ptms:
    print(f"Mean pTM: {np.mean(ptms):.3f}  ({len(ptms)}/{len(natural_records)} sequences had a readable pTM)")
else:
    print("pTM field was not readable in this transformers version -- proceeding on pLDDT only.")
print(f"Collapse rate @ pLDDT<60: {np.mean([r['collapse_60'] for r in natural_records]):.1%}")
print(f"Collapse rate @ pLDDT<70: {np.mean([r['collapse_70'] for r in natural_records]):.1%}")

Loading ESMFold...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/4533 [00:00<?, ?it/s]

EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status     | 
-----------------------------------+------------+-
esm.embeddings.position_ids        | UNEXPECTED | 
esm.contact_head.regression.bias   | MISSING    | 
esm.contact_head.regression.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Folding natural sequences...

Mean pLDDT: 59.25
Mean pTM: 0.286  (400/400 sequences had a readable pTM)
Collapse rate @ pLDDT<60: 53.2%
Collapse rate @ pLDDT<70: 81.0%


In [9]:
# --- Sanity check: does the pLDDT-based label track a known, real correlate of disorder? ---
for r in natural_records:
    r["disorder_score"] = disorder_score(r["sequence"])

collapsed = [r["disorder_score"] for r in natural_records if r["collapse_60"] == 1]
not_collapsed = [r["disorder_score"] for r in natural_records if r["collapse_60"] == 0]

print("=== Label sanity check: composition-based disorder proxy vs. pLDDT collapse label ===")
print(f"Mean disorder score | COLLAPSED   (pLDDT<60): {np.mean(collapsed):+.4f}  (n={len(collapsed)})")
print(f"Mean disorder score | NOT collapsed          : {np.mean(not_collapsed):+.4f}  (n={len(not_collapsed)})")

all_scores = np.array([r["disorder_score"] for r in natural_records])
all_labels = np.array([r["collapse_60"] for r in natural_records])
if np.std(all_scores) > 1e-8 and len(np.unique(all_labels)) > 1:
    corr = np.corrcoef(all_scores, all_labels)[0, 1]
    print(f"Point-biserial correlation (disorder score vs. collapse): r = {corr:+.3f}")
print()
print("If collapsed sequences show a clearly HIGHER disorder score, the pLDDT label is tracking")
print("a real, known biophysical correlate of foldability -- it's behaving sensibly, not noise.")
print("If there's little to no difference, that's a reason to be more skeptical of the label.")

=== Label sanity check: composition-based disorder proxy vs. pLDDT collapse label ===
Mean disorder score | COLLAPSED   (pLDDT<60): +0.1483  (n=213)
Mean disorder score | NOT collapsed          : +0.1428  (n=187)
Point-biserial correlation (disorder score vs. collapse): r = +0.011

If collapsed sequences show a clearly HIGHER disorder score, the pLDDT label is tracking
a real, known biophysical correlate of foldability -- it's behaving sensibly, not noise.
If there's little to no difference, that's a reason to be more skeptical of the label.


In [10]:
# --- Build all feature sets ---
d_model = natural_records[0]["raw_layer_12"].shape[0]
d_latent = natural_records[0]["osae_layer12_full"].shape[0]

X_entropy = np.array([r["entropy_cumsum"] for r in natural_records]).reshape(-1, 1)
X_osae_l12 = np.array([r["osae_layer12_full"][causal_indices] for r in natural_records])
X_raw = {L: np.array([r[f"raw_layer_{L}"] for r in natural_records]) for L in RAW_LAYERS}

print("Feature set shapes:")
print(f"  Entropy baseline:        {X_entropy.shape}")
print(f"  OSAE layer-12 (3 latents): {X_osae_l12.shape}")
for L in RAW_LAYERS:
    print(f"  Raw layer {L}:             {X_raw[L].shape}")

Feature set shapes:
  Entropy baseline:        (400, 1)
  OSAE layer-12 (3 latents): (400, 3)
  Raw layer 6:             (400, 1280)
  Raw layer 12:             (400, 1280)
  Raw layer 18:             (400, 1280)
  Raw layer 24:             (400, 1280)
  Raw layer 30:             (400, 1280)


In [11]:
# --- Repeated 20x 70/30 evaluation across every feature set, for BOTH label thresholds ---
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, precision_score, recall_score

N_REPEATS = 20
TEST_SIZE = 0.3
PCA_COMPONENTS = 20

def fit_eval(X, y, idx_train, idx_test, use_pca=False):
    y_train, y_test = y[idx_train], y[idx_test]
    if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
        return None
    X_train, X_test = X[idx_train], X[idx_test]

    if use_pca and X_train.shape[1] > PCA_COMPONENTS:
        pca = PCA(n_components=PCA_COMPONENTS, random_state=0).fit(X_train)
        X_train, X_test = pca.transform(X_train), pca.transform(X_test)

    scaler = StandardScaler().fit(X_train)
    X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)

    out = {}
    rf = RandomForestClassifier(n_estimators=200, max_depth=4, random_state=0, class_weight="balanced")
    rf.fit(X_train, y_train)
    proba = rf.predict_proba(X_test)[:, 1]
    preds = (proba >= 0.5).astype(int)
    out["RandomForest"] = dict(roc_auc=roc_auc_score(y_test, proba),
                                precision=precision_score(y_test, preds, zero_division=0),
                                recall=recall_score(y_test, preds, zero_division=0))

    svm = SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=0)
    svm.fit(X_train_s, y_train)
    proba = svm.predict_proba(X_test_s)[:, 1]
    preds = (proba >= 0.5).astype(int)
    out["SVM"] = dict(roc_auc=roc_auc_score(y_test, proba),
                       precision=precision_score(y_test, preds, zero_division=0),
                       recall=recall_score(y_test, preds, zero_division=0))
    return out

def run_comparison(y_labels, label_name):
    feature_sets = {"Entropy baseline": (X_entropy, False),
                     "OSAE layer-12 (3 latents)": (X_osae_l12, False)}
    for L in RAW_LAYERS:
        feature_sets[f"Raw layer {L} (PCA-20)"] = (X_raw[L], True)

    splitter = StratifiedShuffleSplit(n_splits=N_REPEATS, test_size=TEST_SIZE, random_state=42)
    results = {name: {"RandomForest": [], "SVM": []} for name in feature_sets}

    for idx_train, idx_test in splitter.split(np.zeros(len(y_labels)), y_labels):
        for name, (X, use_pca) in feature_sets.items():
            r = fit_eval(X, y_labels, idx_train, idx_test, use_pca=use_pca)
            if r is None:
                continue
            for clf_name, metrics in r.items():
                results[name][clf_name].append(metrics)

    print(f"\n=== {N_REPEATS}x repeated 70/30 evaluation -- label: {label_name} (collapse rate {y_labels.mean():.1%}) ===")
    for name in feature_sets:
        for clf_name in ["RandomForest", "SVM"]:
            runs = results[name][clf_name]
            if not runs:
                print(f"[{name:26s} | {clf_name:12s}] no valid runs")
                continue
            aucs = [r["roc_auc"] for r in runs]
            recs = [r["recall"] for r in runs]
            precs = [r["precision"] for r in runs]
            print(f"[{name:26s} | {clf_name:12s}] ROC-AUC={np.mean(aucs):.3f}+/-{np.std(aucs):.3f}  "
                  f"Precision={np.mean(precs):.3f}+/-{np.std(precs):.3f}  Recall={np.mean(recs):.3f}+/-{np.std(recs):.3f}")
    return results

y60 = np.array([r["collapse_60"] for r in natural_records])
y70 = np.array([r["collapse_70"] for r in natural_records])

results_60 = run_comparison(y60, "pLDDT < 60")
results_70 = run_comparison(y70, "pLDDT < 70 (field-standard threshold)")


=== 20x repeated 70/30 evaluation -- label: pLDDT < 60 (collapse rate 53.2%) ===
[Entropy baseline           | RandomForest] ROC-AUC=0.586+/-0.040  Precision=0.584+/-0.029  Recall=0.575+/-0.098
[Entropy baseline           | SVM         ] ROC-AUC=0.581+/-0.068  Precision=0.576+/-0.027  Recall=0.737+/-0.131
[OSAE layer-12 (3 latents)  | RandomForest] ROC-AUC=0.528+/-0.039  Precision=0.562+/-0.043  Recall=0.552+/-0.052
[OSAE layer-12 (3 latents)  | SVM         ] ROC-AUC=0.461+/-0.055  Precision=0.525+/-0.015  Recall=0.897+/-0.118
[Raw layer 6 (PCA-20)       | RandomForest] ROC-AUC=0.711+/-0.034  Precision=0.670+/-0.039  Recall=0.625+/-0.062
[Raw layer 6 (PCA-20)       | SVM         ] ROC-AUC=0.699+/-0.047  Precision=0.663+/-0.036  Recall=0.697+/-0.096
[Raw layer 12 (PCA-20)      | RandomForest] ROC-AUC=0.712+/-0.041  Precision=0.681+/-0.040  Recall=0.634+/-0.070
[Raw layer 12 (PCA-20)      | SVM         ] ROC-AUC=0.709+/-0.051  Precision=0.662+/-0.043  Recall=0.691+/-0.088
[Raw layer 18 

In [12]:
# --- Permutation control: is the best feature set's signal real, or could a classifier +
#     PCA pipeline produce a similar-looking score even on pure noise? ---

def best_feature_and_classifier(results):
    best = None
    for name, per_clf in results.items():
        for clf_name, runs in per_clf.items():
            if not runs:
                continue
            mean_auc = np.mean([r["roc_auc"] for r in runs])
            if best is None or mean_auc > best[2]:
                best = (name, clf_name, mean_auc)
    return best

best_name_60, best_clf_60, best_auc_60 = best_feature_and_classifier(results_60)
print(f"Best feature set at pLDDT<60: {best_name_60} | {best_clf_60} (mean ROC-AUC {best_auc_60:.3f})")

feature_lookup = {"Entropy baseline": (X_entropy, False), "OSAE layer-12 (3 latents)": (X_osae_l12, False)}
for L in RAW_LAYERS:
    feature_lookup[f"Raw layer {L} (PCA-20)"] = (X_raw[L], True)

X_best, use_pca_best = feature_lookup[best_name_60]

rng = np.random.RandomState(99)
shuffled_aucs = []
splitter = StratifiedShuffleSplit(n_splits=N_REPEATS, test_size=TEST_SIZE, random_state=42)
for idx_train, idx_test in splitter.split(np.zeros(len(y60)), y60):
    y_shuffled = y60.copy()
    rng.shuffle(y_shuffled)
    r = fit_eval(X_best, y_shuffled, idx_train, idx_test, use_pca=use_pca_best)
    if r is not None and best_clf_60 in r:
        shuffled_aucs.append(r[best_clf_60]["roc_auc"])

print(f"\n=== PERMUTATION CONTROL: {best_name_60} | {best_clf_60} ===")
print(f"Real labels    -> mean ROC-AUC {best_auc_60:.3f}")
if shuffled_aucs:
    print(f"Shuffled labels -> mean ROC-AUC {np.mean(shuffled_aucs):.3f} +/- {np.std(shuffled_aucs):.3f}  (n={len(shuffled_aucs)})")
    print()
    print("Shuffled labels have no real relationship to the features by construction, so this")
    print("should land close to 0.50. If it does, the real result above is genuine pattern-")
    print("matching, not an artifact of the classifier or PCA step being able to fit anything.")
    print("If the shuffled score is also well above 0.50, that would be a red flag worth")
    print("investigating before trusting the real result.")
else:
    print("No valid shuffled splits produced.")

Best feature set at pLDDT<60: Raw layer 30 (PCA-20) | RandomForest (mean ROC-AUC 0.772)

=== PERMUTATION CONTROL: Raw layer 30 (PCA-20) | RandomForest ===
Real labels    -> mean ROC-AUC 0.772
Shuffled labels -> mean ROC-AUC 0.496 +/- 0.044  (n=20)

Shuffled labels have no real relationship to the features by construction, so this
should land close to 0.50. If it does, the real result above is genuine pattern-
matching, not an artifact of the classifier or PCA step being able to fit anything.
If the shuffled score is also well above 0.50, that would be a red flag worth
investigating before trusting the real result.


In [13]:
print("=== FINAL VALIDATION VERDICT ===")
print("Three things to read off the results above before writing begins:")
print()
print("1. Does a raw deep layer (24 or 30) still clearly beat the entropy baseline, in both")
print("   classifiers, at both thresholds, at this bigger N=400? Compare against the two")
print("   earlier N=200 runs (layer 24 was ~0.73-0.77 vs baseline ~0.56-0.66 at the pLDDT<60")
print("   bar, both times) -- a third consistent result here makes this a genuinely solid,")
print("   three-times-reproduced finding.")
print("2. Where does the signal peak across the five layers? If it keeps climbing from 6 to")
print("   30, deeper still might help; if it plateaus or dips at 30, that pins down roughly")
print("   where in the network the useful signal actually lives.")
print("3. Did the permutation control collapse to ~0.50? If yes, that's the rigor check that")
print("   closes off the 'maybe the classifier is just fitting noise' objection before a")
print("   reviewer can raise it.")
print()
print("This is the last planned experiment. Whatever it shows, it's time to write.")

=== FINAL VALIDATION VERDICT ===
Three things to read off the results above before writing begins:

1. Does a raw deep layer (24 or 30) still clearly beat the entropy baseline, in both
   classifiers, at both thresholds, at this bigger N=400? Compare against the two
   earlier N=200 runs (layer 24 was ~0.73-0.77 vs baseline ~0.56-0.66 at the pLDDT<60
   bar, both times) -- a third consistent result here makes this a genuinely solid,
   three-times-reproduced finding.
2. Where does the signal peak across the five layers? If it keeps climbing from 6 to
   30, deeper still might help; if it plateaus or dips at 30, that pins down roughly
   where in the network the useful signal actually lives.
3. Did the permutation control collapse to ~0.50? If yes, that's the rigor check that
   closes off the 'maybe the classifier is just fitting noise' objection before a
   reviewer can raise it.

This is the last planned experiment. Whatever it shows, it's time to write.


## Addendum: risk-coverage / selective-prediction analysis (added 2026-08-20)

This re-derives the abstention/risk-coverage analysis referenced in project memory as
"Finding 5" (97.5% precision at the top decile), which was never actually saved to a
notebook file (see `notes/results.md` SS3c). It reuses the exact best feature set/model
identified above (`X_best`, `y60`, `best_clf_60`) so the number is directly comparable to
the headline ROC-AUC already reported, and computes proper **out-of-fold** predicted
probabilities (`cross_val_predict`, `StratifiedKFold`) rather than reusing in-sample
predictions, to avoid leakage.

**Terminology note (see `notes/context-and-decisions.md` SS2):** this is an *ordinary*
classifier evaluation on out-of-fold probabilities. It is **not** a distribution-free or
conformal-calibration guarantee -- do not describe it that way in the paper unless an
actual conformal calibration step (Angelopoulos & Bates, arXiv:2107.07511) is added on
top of this.

In [16]:
# --- Finding 5 addendum: risk-coverage / selective-prediction analysis ---
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline

def out_of_fold_proba(X, y, n_splits=10, use_pca=True, random_state=0):
    steps = [("scaler", StandardScaler())]
    if use_pca and X.shape[1] > PCA_COMPONENTS:
        steps.append(("pca", PCA(n_components=PCA_COMPONENTS, random_state=random_state)))
    steps.append(("rf", RandomForestClassifier(n_estimators=200, max_depth=4,
                                                 random_state=random_state, class_weight="balanced")))
    pipe = Pipeline(steps)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    proba = cross_val_predict(pipe, X, y, cv=skf, method="predict_proba")[:, 1]
    return proba

# Reuse the best feature set identified above (raw layer 30, PCA-20) against the pLDDT<60
# label, so this analysis is on the exact model already reported in SS3a of results.md.
proba_oof = out_of_fold_proba(X_best, y60, n_splits=10, use_pca=use_pca_best)
preds_oof = (proba_oof >= 0.5).astype(int)
confidence = np.abs(proba_oof - 0.5) * 2  # 0 = coin flip, 1 = maximally confident

print(f"Out-of-fold ROC-AUC ({best_name_60} | RandomForest, stratified 10-fold): "
      f"{roc_auc_score(y60, proba_oof):.3f}")
print(f"(compare to the {N_REPEATS}x-repeated-split estimate above: {best_auc_60:.3f} -- "
      f"should be in the same ballpark; this is a single stratified 10-fold pass instead)")

# --- Risk-coverage curve: sort by confidence, descending; at each coverage level,
#     report the error rate (risk) among the covered (highest-confidence) subset,
#     plus precision restricted to the predicted-collapse calls within that subset. ---
order = np.argsort(-confidence)
sorted_correct = (preds_oof[order] == y60[order]).astype(int)
n = len(y60)

coverage_levels = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
print("Coverage  N_covered  Risk(error rate)  Precision(predicted-collapse, covered)")
for cov in coverage_levels:
    k = max(1, int(round(cov * n)))
    covered_correct = sorted_correct[:k]
    covered_preds = preds_oof[order][:k]
    covered_true = y60[order][:k]
    risk = 1 - covered_correct.mean()
    pos_mask = covered_preds == 1
    precision = covered_true[pos_mask].mean() if pos_mask.sum() > 0 else float("nan")
    print(f"{cov:6.0%}    {k:5d}      {risk:6.3f}            {precision:6.3f}  (n_pos_pred={pos_mask.sum()})")

print("Top-decile precision (most confident 10% of predictions, predicted-collapse calls only):")
k10 = max(1, int(round(0.1 * n)))
top10_preds = preds_oof[order][:k10]
top10_true = y60[order][:k10]
pos_mask10 = top10_preds == 1
if pos_mask10.sum() > 0:
    print(f"  precision = {top10_true[pos_mask10].mean():.3f}  "
          f"(n={pos_mask10.sum()} predicted-collapse calls in top decile)")
else:
    print("  no predicted-collapse calls in the top decile -- report the coverage/risk table above instead")

print("This is the real, file-backed replacement for the memory-only 'Finding 5' claim.")
print("Whatever the top-decile precision number is above, that is the number to cite -- not")
print("the unverified 97.5% figure from prior chat memory, which could not be located in any")
print("notebook file (see notes/results.md SS3c).")

Out-of-fold ROC-AUC (Raw layer 30 (PCA-20) | RandomForest, stratified 10-fold): 0.767
(compare to the 20x-repeated-split estimate above: 0.772 -- should be in the same ballpark; this is a single stratified 10-fold pass instead)
Coverage  N_covered  Risk(error rate)  Precision(predicted-collapse, covered)
   10%       40       0.050             0.958  (n_pos_pred=24)
   20%       80       0.088             0.889  (n_pos_pred=45)
   30%      120       0.108             0.877  (n_pos_pred=65)
   40%      160       0.156             0.830  (n_pos_pred=88)
   50%      200       0.180             0.827  (n_pos_pred=110)
   60%      240       0.208             0.818  (n_pos_pred=121)
   70%      280       0.261             0.774  (n_pos_pred=137)
   80%      320       0.287             0.747  (n_pos_pred=162)
   90%      360       0.292             0.742  (n_pos_pred=182)
  100%      400       0.307             0.721  (n_pos_pred=204)
Top-decile precision (most confident 10% of predictions, p